# Day 09. Exercise 02
# Metrics

## 0. Imports

In [64]:
import time

from typing import Any, Dict, List, Union
import pandas as pd
import joblib
from IPython.core.display import Markdown
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

In [65]:
notebook_start_time = time.time()

## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.

In [66]:
df = pd.read_csv("../data/day-of-week-not-scaled.csv")
df_days = pd.read_csv("../data/dayofweek.csv")
df["dayofweek"] = df_days["dayofweek"].values

X = df.drop(columns="dayofweek")
y = df["dayofweek"]

2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [67]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21, stratify=y
)

## 2. SVM

1. Use the best parameters from the previous exercise and train the model of SVM.

In [68]:
svc_best_params = {
    "C": 10,
    "class_weight": None,
    "gamma": "auto",
    "kernel": "rbf",
    "probability": True,
    "random_state": 21,
}

svc_model = SVC(**svc_best_params)
svc_model.fit(X_train, y_train)
y_pred = svc_model.predict(X_test)

2. You need to calculate `accuracy`, `precision`, `recall`, `ROC AUC`.

 - `precision` and `recall` should be calculated for each class (use `average='weighted'`)
 - `ROC AUC` should be calculated for each class against any other class (all possible pairwise combinations) and then weighted average should be applied for the final metric
 - the code in the cell should display the result as below:

```
accuracy is 0.88757
precision is 0.89267
recall is 0.88757
roc_auc is 0.97878
```

In [69]:
acc = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="weighted")
recall = recall_score(y_test, y_pred, average="weighted")
y_prob = svc_model.predict_proba(X_test)
roc_auc = roc_auc_score(y_test, y_prob, multi_class="ovo", average="weighted")

In [70]:
print(f"accuracy is {acc:.5f}")
print(f"precision is {precision:.5f}")
print(f"recall is {recall:.5f}")
print(f"roc_auc is {roc_auc:.5f}")

accuracy is 0.88757
precision is 0.89267
recall is 0.88757
roc_auc is 0.97878


## 3. Decision tree

1. The same task for decision tree

In [71]:
dt_best_params = {
    "class_weight": "balanced",
    "criterion": "gini",
    "max_depth": 21,
    "random_state": 21,
}

dt_model = DecisionTreeClassifier(**dt_best_params)
dt_model.fit(X_train, y_train)
y_predDT = dt_model.predict(X_test)

In [72]:
acc_dt = accuracy_score(y_test, y_predDT)
precision_dt = precision_score(y_test, y_predDT, average="weighted")
recall_dt = recall_score(y_test, y_predDT, average="weighted")
y_prob_dt = dt_model.predict_proba(X_test)
roc_auc_dt = roc_auc_score(y_test, y_prob_dt, multi_class="ovo", average="weighted")

In [73]:
print(f"accuracy is {acc_dt:.5f}")
print(f"precision is {precision_dt:.5f}")
print(f"recall is {recall_dt:.5f}")
print(f"roc_auc is {roc_auc_dt:.5f}")

accuracy is 0.88462
precision is 0.88765
recall is 0.88462
roc_auc is 0.93528


## 4. Random forest

1. The same task for random forest.

In [74]:
rf_best_params = {
    "class_weight": "balanced",
    "criterion": "entropy",
    "max_depth": 24,
    "n_estimators": 100,
    "random_state": 21,
}

rf_model = RandomForestClassifier(**rf_best_params)
rf_model.fit(X_train, y_train)
y_predRF = rf_model.predict(X_test)

In [75]:
acc_rf = accuracy_score(y_test, y_predRF)
precision_rf = precision_score(y_test, y_predRF, average="weighted")
recall_rf = recall_score(y_test, y_predRF, average="weighted")
y_prob_rf = rf_model.predict_proba(X_test)
roc_auc_rf = roc_auc_score(y_test, y_prob_rf, multi_class="ovo", average="weighted")

In [76]:
print(f"accuracy is {acc_rf:.5f}")
print(f"precision is {precision_rf:.5f}")
print(f"recall is {recall_rf:.5f}")
print(f"roc_auc is {roc_auc_rf:.5f}")

accuracy is 0.92604
precision is 0.92754
recall is 0.92604
roc_auc is 0.98939


In [77]:
summary_df = pd.DataFrame(
    {
        "Model": ["svc_model", "dt_mode", "rf_model"],
        "accuracy": [acc, acc_dt, acc_rf],
        "precision": [precision, precision_dt, precision_rf],
        "recall": [recall, recall_dt, recall_rf],
        "roc auc": [roc_auc, roc_auc_dt, roc_auc_rf],
        "params": [svc_model, dt_model, rf_model],
    }
)
(
    summary_df.drop("params", axis=1)
    .style.highlight_max(
        color="lightgreen",
        axis=0,
    )
    .format(
        {
            "accuracy": "{:.5f}",
            "precision": "{:.5f}",
            "recall": "{:.5f}",
            "roc auc": "{:.5f}",
        }
    )
    .hide_index()
)


Model,accuracy,precision,recall,roc auc
svc_model,0.88757,0.89267,0.88757,0.97878
dt_mode,0.88462,0.88765,0.88462,0.93528
rf_model,0.92604,0.92754,0.92604,0.98939


## 5. Predictions

1. Choose the best model.
2. Analyze: for which `weekday` your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which `labname` and for which `users`.
3. Save the model.

In [78]:
summary_df.loc[2, "params"]

RandomForestClassifier(class_weight='balanced', criterion='entropy',
                       max_depth=24, random_state=21)

In [79]:
test_df = X_test.copy()
test_df["y_test"] = y_test
test_df["y_pred"] = pd.Series(y_predRF, index=y_test.index)
test_errors_df = test_df[test_df["y_test"] != test_df["y_pred"]]
test_errors_df.head(3);

In [80]:
test_errors_df.filter(like="uid_user_").head(3);

In [81]:
user_errors_data = test_errors_df.filter(like="uid_user_").sum(axis=0)
user_errors_data.head(3);

In [82]:
all_users_data = X.filter(like="uid_user_").sum(axis=0)

In [83]:
user_errors_percent = user_errors_data / all_users_data * 100
user_errors_percent.head(3);

In [84]:
lab_errors_data = test_errors_df.filter(like="labname_").sum(axis=0)

In [85]:
all_labs_data = X.filter(like="labname_").sum(axis=0)

In [86]:
lab_errors_percent = lab_errors_data / all_labs_data * 100
lab_errors_percent.head(3);

In [87]:
worst_user = user_errors_percent.idxmax()
worst_user_val = user_errors_percent.max()

In [88]:
worst_lab = lab_errors_percent.idxmax()
worst_lab_val = lab_errors_percent.max()

In [89]:
weekday_errors = test_errors_df["y_test"].value_counts()
weekday_all = df["dayofweek"].value_counts()
weekday_errors_percent = weekday_errors / weekday_all * 100

weekday_df = (
    weekday_errors.to_frame(name="prediction_errors")
    .rename_axis("weekday")
    .assign(all_predictions=weekday_all, errors_percent=weekday_errors_percent)
    .sort_index()
)

days_dict = {0: "Пн", 1: "Вт", 2: "Ср", 3: "Чт", 4: "Пт", 5: "Сб", 6: "Вс"}
weekday_df.index = weekday_df.index.map(days_dict)

weekday_df.style.highlight_max(
    subset=["errors_percent"], color="lightgreen", axis=0
).format({"errors_percent": "{:.2f}"})

,prediction_errors,all_predictions,errors_percent
weekday,,,
Пн,6,136,4.41
Вт,4,274,1.46
Ср,2,149,1.34
Чт,3,396,0.76
Пт,3,104,2.88
Сб,5,271,1.85
Вс,2,356,0.56


In [90]:
days_dict = {0: "Пн", 1: "Вт", 2: "Ср", 3: "Чт", 4: "Пт", 5: "Сб", 6: "Вс"}
weekday_errors_percent.index = weekday_errors_percent.index.map(days_dict).fillna(0)
worst_day = weekday_errors_percent.idxmax()
worst_day_val = weekday_errors_percent.max()

In [91]:
print(f"Больше всего ошибок модель совершает в: {worst_day} ({worst_day_val:.2f}%)")
print(f"Больше всего ошибок модель совершает: {worst_user} ({worst_user_val:.2f}%)")
print(f"Лаба с наиболшим количеством ошибок: {worst_lab} ({worst_lab_val:.2f}%)")

Больше всего ошибок модель совершает в: Пн (4.41%)
Больше всего ошибок модель совершает: uid_user_6 (16.67%)
Лаба с наиболшим количеством ошибок: labname_lab03 (100.00%)


In [92]:
joblib.dump(
    summary_df.loc[2, "params"], "../data/best_random_forest_model_ex02.joblib"
);

## 6. Function

1. Write a function that takes a list of different models and a corresponding list of parameters (dicts) and returns a dict that contains all the 4 metrics for each model.

In [93]:
def calculate_model_metrics(models: list, params: List[Dict]):
    params_dct = {}
    for model_class, model_params in zip(models, params):
        model = model_class(**model_params)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average="weighted")
        recall = recall_score(y_test, y_pred, average="weighted")

        y_prob = model.predict_proba(X_test)
        roc_auc = roc_auc_score(y_test, y_prob, multi_class="ovo", average="weighted")

        dct = {
            "accuracy": round(acc, 5),
            "precision": round(precision, 5),
            "recall": round(recall, 5),
            "roc_auc": round(roc_auc, 5),
        }

        params_dct[model_class.__name__] = dct

    return params_dct

In [94]:
models = [SVC, DecisionTreeClassifier, RandomForestClassifier]
params = [
    {
        "C": 10,
        "class_weight": None,
        "gamma": "auto",
        "kernel": "rbf",
        "probability": True,
        "random_state": 21,
    },
    {
        "class_weight": "balanced",
        "criterion": "gini",
        "max_depth": 21,
        "random_state": 21,
    },
    {
        "class_weight": "balanced",
        "criterion": "entropy",
        "max_depth": 24,
        "n_estimators": 100,
        "random_state": 21,
    },
]

for k, inner_dct in calculate_model_metrics(models, params).items():
    print(f"Параметры для модели {k}:")
    for key, val in inner_dct.items():
        print(f"{key}: {val}")
    print()

Параметры для модели SVC:
accuracy: 0.88757
precision: 0.89267
recall: 0.88757
roc_auc: 0.97878

Параметры для модели DecisionTreeClassifier:
accuracy: 0.88462
precision: 0.88765
recall: 0.88462
roc_auc: 0.93528

Параметры для модели RandomForestClassifier:
accuracy: 0.92604
precision: 0.92754
recall: 0.92604
roc_auc: 0.98939



In [95]:
print(
    calculate_model_metrics(
        [SVC],
        [
            {
                "C": 10,
                "class_weight": None,
                "gamma": "auto",
                "kernel": "rbf",
                "probability": True,
                "random_state": 21,
            }
        ],
    )
)

{'SVC': {'accuracy': 0.88757, 'precision': 0.89267, 'recall': 0.88757, 'roc_auc': 0.97878}}


In [96]:
# хотелось бы слегка отойти от ТЗ и иметь возм-ть передавать просто класс модели и ее словарь параметров в случае, если она только одна.


def calculate_model_metrics_alt(
    models: Union[list, Any], params: Union[List[Dict], dict]
) -> dict:

    if not isinstance(models, list):
        models = [models]
    if not isinstance(params, list):
        params = [params]

    params_dct = {}
    for model_class, model_params in zip(models, params):
        model = model_class(**model_params)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
        recall = recall_score(y_test, y_pred, average="weighted")

        y_prob = model.predict_proba(X_test)
        roc_auc = roc_auc_score(y_test, y_prob, multi_class="ovo", average="weighted")

        dct = {
            "accuracy": round(acc, 5),
            "precision": round(precision, 5),
            "recall": round(recall, 5),
            "roc_auc": round(roc_auc, 5),
        }

        params_dct[model_class.__name__] = dct

    return params_dct

In [97]:
print(
    calculate_model_metrics_alt(
        SVC,
        {
            "C": 10,
            "class_weight": None,
            "gamma": "auto",
            "kernel": "rbf",
            "probability": True,
            "random_state": 21,
        },
    )
)

{'SVC': {'accuracy': 0.88757, 'precision': 0.89267, 'recall': 0.88757, 'roc_auc': 0.97878}}


In [98]:
notebook_total_time = time.time() - notebook_start_time
minutes = int(notebook_total_time // 60)
seconds = int(notebook_total_time % 60)

display(
    Markdown(
        f"""
---
**Время работы блокнота:** <span style="color:green">**{minutes} мин. {seconds} сек.**</span>

---
"""
    )
)


---
**Время работы блокнота:** <span style="color:green">**0 мин. 4 сек.**</span>

---
